# Pokémon TCG AI — train + evaluate on Kaggle T4×2

**What this notebook does, in order:**

1. Verifies T4×2 GPU is visible.
2. Installs the repo (two paths: from a GitHub URL, or from a Kaggle dataset).
3. Runs the test + lint + benchmark sanity battery.
4. Builds the **stacked aggro deck** (the corrected eval fixture; see `ROOT_CAUSE_ANALYSIS.md`).
5. Trains a small AlphaZero loop on GPU → **produces a checkpoint** (the single biggest item missing from the previous submission).
6. Evaluates: agent-with-checkpoint **vs random** AND **vs heuristic-MCTS** on the stacked deck. This produces the missing win-rate number.
7. Saves everything (checkpoint, CSVs, JSON summary, figures) under `/kaggle/working/` so it can be downloaded and attached to the Kaggle Writeup.

**Hardware required:** Settings → Accelerator → **GPU T4×2**. Internet **must be on** for the GitHub install path.

**Wall-clock budget:** ~45–90 min on T4×2 for the default knobs in §5 below.

## 0. Configuration — edit me before running

In [ ]:
# === User-tunable knobs ===
REPO_GIT_URL = "https://github.com/arnav-chauhan-kgpian/PTCG.git"
REPO_GIT_BRANCH = "main"
KAGGLE_DATASET_REPO_PATH = "/kaggle/input/ptcg"  # used if git clone fails

# Training knobs — keep small first; bump after one successful run
TRAINING_ROUNDS = 4               # full pipeline rounds (self-play + train + arena + promote)
SELFPLAY_GAMES_PER_ROUND = 16     # parallelisable
MCTS_ITERATIONS_SELFPLAY = 32     # higher = better data, slower
MCTS_ITERATIONS_ARENA = 16        # arena play during promotion
TRAINER_BATCH_SIZE = 64
TRAINER_STEPS_PER_ROUND = 200
REPLAY_CAPACITY = 50_000
EARLY_STOP_MAX_WALL_S = 60 * 60   # 60-minute training budget

# Evaluation knobs
EVAL_N_GAMES_AGENT_VS_RANDOM = 40
EVAL_N_GAMES_AGENT_VS_HEURISTIC = 20
EVAL_MAX_ACTIONS_PER_GAME = 400
EVAL_AGENT_ITERATIONS = 32

## 1. GPU check

In [ ]:
import os, sys, subprocess, json, time, math, random, pathlib
print("python:", sys.version.split()[0])
subprocess.run(["nvidia-smi", "-L"], check=False)
import torch
print("torch:", torch.__version__, "cuda:", torch.cuda.is_available(),
      "devices:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  cuda:{i} →", torch.cuda.get_device_name(i))

# Silence loguru DEBUG spam from the effect-matcher.
# (POKEMON_AI_LOG_LEVEL is only read inside src/server/app.py::create_app,
#  not by the CLI or library code — so we have to do it directly here.)
os.environ["POKEMON_AI_LOG_LEVEL"] = "ERROR"
try:
    from loguru import logger as _loguru
    _loguru.remove()
    _loguru.add(sys.stderr, level="ERROR")
    print("loguru sink: stderr @ ERROR")
except ImportError:
    pass

## 2. Acquire the repo

Tries `git clone` first; falls back to the Kaggle dataset path. **If neither works, upload your repo as a Kaggle dataset and set `KAGGLE_DATASET_REPO_PATH` above.**

In [ ]:
REPO_DIR = "/kaggle/working/PTCG"

def have_repo():
    return (pathlib.Path(REPO_DIR) / "src" / "cli" / "main.py").exists()

if not have_repo():
    if REPO_GIT_URL:
        print(f"Trying git clone {REPO_GIT_URL} ...")
        rc = subprocess.run(
            ["git", "clone", "--depth", "1", "-b", REPO_GIT_BRANCH, REPO_GIT_URL, REPO_DIR],
            check=False,
        ).returncode
        if rc != 0:
            print("git clone failed; trying Kaggle dataset path")
    if not have_repo() and pathlib.Path(KAGGLE_DATASET_REPO_PATH).exists():
        subprocess.run(["cp", "-r", KAGGLE_DATASET_REPO_PATH, REPO_DIR], check=True)

assert have_repo(), (
    f"Repo not found at {REPO_DIR}. Either: "
    f"(a) ensure Internet is on (Settings → Internet → On) so git clone works, OR "
    f"(b) upload the repository as a Kaggle dataset and point KAGGLE_DATASET_REPO_PATH at it."
)
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())
print(subprocess.run(["ls"], capture_output=True, text=True).stdout)

## 3. Install

In [ ]:
# pin setuptools first (the project uses pyproject.toml-based editable install)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                "--upgrade", "pip", "setuptools", "wheel"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                "-e", ".[dev,server]"], check=True)
# Kaggle base image ships torch with CUDA already — do NOT overwrite it.
print("install OK")
sys.path.insert(0, REPO_DIR)

## 4. Smoke tests + benchmark

In [ ]:
os.environ["POKEMON_AI_LOG_LEVEL"] = "ERROR"
# Lint
subprocess.run([sys.executable, "-m", "ruff", "check", "src", "tests"], check=False)
# Fast unit-test smoke
subprocess.run([sys.executable, "-m", "pytest", "-q", "--maxfail=5",
                "-x", "tests/server", "tests/competition", "tests/cli"], check=False)
# Benchmark — parse the full JSON document, not just the last line.
bench = subprocess.run([sys.executable, "-m", "src.cli", "--json", "benchmark"],
                        capture_output=True, text=True, check=True)
raw = bench.stdout.strip()
# Find the first '{' to skip any non-JSON preamble (loguru warnings, etc.)
i = raw.find("{")
j = raw.rfind("}")
assert i != -1 and j != -1, f"no JSON found in benchmark output:\n{raw[:500]}"
BENCH = json.loads(raw[i:j+1])
print(json.dumps(BENCH, indent=2))
BENCH

## 5. Train an AlphaZero loop on GPU → produces a checkpoint

This is the single biggest item missing from the original submission. Runs `TrainingPipeline.run()` with conservative knobs that fit in T4×2 wall-clock.

In [ ]:
from src.cards import load_repository
from src.simulator import PokemonTCGSimulator
from src.training.config import (
    PipelineConfig, SelfPlayConfig, ArenaConfig, PromotionConfig, EarlyStoppingConfig,
)
from src.training.pipeline import TrainingPipeline
from src.mcts.node import MCTSAction

repo = load_repository()
print(f"repository: {len(repo.list_all())} cards")

# Build the corrected stacked aggro deck (see ROOT_CAUSE_ANALYSIS.md)
def build_stacked_deck(repo):
    from src.cards.enums import Stage
    from src.cards.models import EnergyCard, PokemonCard
    cands = []
    for c in repo.list_all():
        if isinstance(c, PokemonCard) and c.stage == Stage.BASIC and c.attacks:
            for att in c.attacks:
                cost_n = att.cost.total_count if att.cost else 99
                if cost_n <= 2 and att.damage and att.damage.base >= 20:
                    cands.append((c, cost_n, att.damage.base))
                    break
    cands.sort(key=lambda x: (x[1], -x[2]))
    top5 = [c for c, _, _ in cands[:5]]
    deck: list[int] = []
    for c in top5:
        deck.extend([c.card_id] * 4)
    energies = [c for c in repo.list_all() if isinstance(c, EnergyCard)]
    basic = next((e for e in energies if "Basic" in (e.name or "")), energies[0])
    deck.extend([basic.card_id] * (60 - len(deck)))
    return deck[:60], [c.name for c in top5], basic.name

DECK, TOP5, ENERGY = build_stacked_deck(repo)
print("deck:", len(DECK), "cards /", len(set(DECK)), "unique")
print("attackers:", TOP5, " energy:", ENERGY)

In [ ]:
# Build an action_map sample: collect legal actions from a few opening states.
sim = PokemonTCGSimulator(repo, seed=0)
action_set: set = set()
for s in range(8):
    sim2 = PokemonTCGSimulator(repo, seed=s)
    st = sim2.start_game(DECK, DECK)
    for _ in range(64):
        legal = sim2.legal_actions(st)
        if not legal:
            break
        for a in legal:
            action_set.add((a.action_type, tuple(sorted(a.details.items())) if isinstance(a.details, dict) else a.details))
        st = sim2.apply_action(st, legal[0])
        if sim2.is_terminal(st):
            break
action_map = []
for at, details in sorted(action_set, key=lambda x: x[0]):
    details_dict = dict(details) if isinstance(details, tuple) else {}
    action_map.append(MCTSAction(action_type=at, details=details_dict))
print("action_map size:", len(action_map))

In [ ]:
EXP_DIR = pathlib.Path("/kaggle/working/experiments")
EXP_DIR.mkdir(parents=True, exist_ok=True)

from src.training.curriculum import make_curriculum
from dataclasses import replace

cfg = PipelineConfig(
    rounds=TRAINING_ROUNDS,
    experiment_dir=str(EXP_DIR),
    experiment_name="kaggle_t4x2_run",
    seed=42,
    replay_capacity=REPLAY_CAPACITY,
    selfplay=SelfPlayConfig(
        games_per_round=SELFPLAY_GAMES_PER_ROUND,
        mcts_iterations=MCTS_ITERATIONS_SELFPLAY,
        temperature_high=1.0,
        temperature_low=0.0,
    ),
    arena=ArenaConfig(
        n_games=20,
        mcts_iterations=MCTS_ITERATIONS_ARENA,
        seed=7,
    ),
    promotion=PromotionConfig(win_rate_threshold=0.55),
    early_stopping=EarlyStoppingConfig(
        max_wall_clock_s=EARLY_STOP_MAX_WALL_S,
    ),
)

# Critical: the network's action_size must match the action_map length,
# otherwise the policy head emits 256-wide logits while training targets
# are len(action_map)-wide → tensor-size mismatch in loss.
cfg = replace(cfg, network=replace(
    cfg.network,
    action_size=len(action_map),
    device="cuda" if torch.cuda.is_available() else "cpu",
))
print("network device:    ", cfg.network.device)
print("network action_size:", cfg.network.action_size,
      " (action_map size:", len(action_map), ")")

# Pass an explicit curriculum that starts each round with the stacked deck.
# Without this the pipeline would use default_curriculum() — which returns
# the empty GameState.new_game() and the pipeline now refuses to run it
# (previously this silently produced 0 training samples per round).
sim_for_pipeline = PokemonTCGSimulator(repo, seed=42)
curriculum = make_curriculum(sim_for_pipeline, DECK, DECK,
                              name="stacked_aggro_mirror",
                              description="Mirror match on the stacked aggro deck")

pipeline = TrainingPipeline(
    simulator=sim_for_pipeline,
    action_map=action_map,
    config=cfg,
    curriculum=curriculum,
)
t0 = time.perf_counter()
result = pipeline.run()
print(f"training done in {time.perf_counter()-t0:.0f}s")
print("rounds_completed:", result.rounds_completed,
      "promotions:", result.promotions,
      "best_checkpoint:", result.best_checkpoint)

In [ ]:
# Locate the produced checkpoint
from src.mcts.checkpoints import CheckpointManager
ckpt_root = pathlib.Path(result.experiment_summary.get("checkpoints_dir")) \
            if result.experiment_summary.get("checkpoints_dir") \
            else EXP_DIR / "kaggle_t4x2_run" / "checkpoints"
ckpts = sorted(ckpt_root.glob("ckpt_*.pt"))
print("found checkpoints:", [p.name for p in ckpts])
ASSERT_CKPT = ckpts[-1] if ckpts else None
print("using:", ASSERT_CKPT)

## 6. Evaluation — the missing head-to-head numbers

Three matches against the stacked deck:
* **A.** Agent (with trained checkpoint) vs Random
* **B.** Agent (with trained checkpoint) vs Heuristic-MCTS (no neural)
* **C.** Mirror sanity (Agent vs Agent)

In [ ]:
import csv, statistics
from src.competition.agent import AgentConfig, CompetitionAgent
from src.competition.inference_engine import InferenceEngine
from src.mcts.checkpoints import CheckpointManager
from src.game_state.zones import GameStatus

def wilson_ci(k, n, z=1.96):
    if n == 0:
        return (0.0, 1.0)
    p = k / n
    d = 1 + z*z/n
    c = (p + z*z/(2*n))/d
    h = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n))/d
    return (max(0.0, c-h), min(1.0, c+h))

def play(sim, deck, pa, pb, max_actions, seed):
    rng = random.Random(seed)
    state = sim.start_game(deck, deck)
    n_actions = 0
    lats = []
    while n_actions < max_actions and not sim.is_terminal(state):
        legal = sim.legal_actions(state)
        if not legal:
            break
        pid = state.current_player
        t0 = time.perf_counter()
        a = (pa if pid == 0 else pb)(state, legal, rng)
        if pid == 0:
            lats.append((time.perf_counter() - t0) * 1000.0)
        state = sim.apply_action(state, a)
        n_actions += 1
    winner = (
        0 if state.game_status == GameStatus.PLAYER_0_WIN
        else 1 if state.game_status == GameStatus.PLAYER_1_WIN
        else "draw" if state.game_status == GameStatus.DRAW
        else "timeout"
    )
    return dict(
        winner=winner, actions=n_actions, final_turn=state.turn_number,
        p0_prizes_left=state.players[0].prizes_remaining,
        p1_prizes_left=state.players[1].prizes_remaining,
        knockouts=len(state.knockout_history),
        avg_ms_p0=round(statistics.mean(lats), 2) if lats else 0.0,
        max_ms_p0=round(max(lats), 2) if lats else 0.0,
    )

def random_pol(state, legal, rng):
    return rng.choice(legal)

def make_agent_pol(agent):
    def pol(state, legal, rng):
        a = agent.choose_action(state)
        return a if a is not None else rng.choice(legal)
    return pol


def build_trained_agent(ckpt_path, repo, iterations, device):
    """Load a training-pipeline checkpoint and wrap it into a CompetitionAgent.

    Uses CheckpointManager.load (which understands the {model_state, metadata}
    schema written by TrainingPipeline) rather than CompetitionAgent.load
    (which routes through NetworkWrapper.load and expects the older
    {config, model_state} schema).
    """
    mgr = CheckpointManager(ckpt_path.parent)
    network, _meta = mgr.load(ckpt_path.stem, device=device)
    inference = InferenceEngine(network)
    return CompetitionAgent(
        simulator=PokemonTCGSimulator(repo, seed=42),
        inference=inference,
        config=AgentConfig(iterations=iterations, time_budget_s=2.0),
    )

device = "cuda" if torch.cuda.is_available() else "cpu"

# Build the three agents
if ASSERT_CKPT is not None:
    agent_trained   = build_trained_agent(ASSERT_CKPT, repo, EVAL_AGENT_ITERATIONS, device)
    agent_trained_b = build_trained_agent(ASSERT_CKPT, repo, EVAL_AGENT_ITERATIONS, device)
else:
    print("⚠ no checkpoint produced — falling back to heuristic for 'trained' role")
    agent_trained = CompetitionAgent.load(
        None, repository=repo,
        config=AgentConfig(iterations=EVAL_AGENT_ITERATIONS, time_budget_s=2.0),
    )
    agent_trained_b = CompetitionAgent.load(
        None, repository=repo,
        config=AgentConfig(iterations=EVAL_AGENT_ITERATIONS, time_budget_s=2.0),
    )

# Heuristic agent always uses the no-checkpoint path
agent_heur = CompetitionAgent.load(
    None, repository=repo,
    config=AgentConfig(iterations=EVAL_AGENT_ITERATIONS, time_budget_s=2.0),
)

print("agents constructed.")
print("  trained.inference is not None:", agent_trained.inference is not None)
print("  heuristic.inference is not None:", agent_heur.inference is not None)

In [ ]:
def run_match(label, deck, pa, pb, n_games, max_actions):
    print(f"\n=== {label}  n={n_games}")
    rows = []
    for i in range(n_games):
        sim = PokemonTCGSimulator(repo, seed=500 + i)
        r = play(sim, deck, pa, pb, max_actions=max_actions, seed=600 + i)
        rows.append({"game_id": i, **r})
        print(f"  g{i}: winner={r['winner']!s:<7} actions={r['actions']:>3} "
              f"prizes=({r['p0_prizes_left']},{r['p1_prizes_left']}) "
              f"kos={r['knockouts']} avg_ms={r['avg_ms_p0']}")
    return rows

def summarize(label, rows):
    n = len(rows)
    p0 = sum(1 for r in rows if r["winner"] == 0)
    p1 = sum(1 for r in rows if r["winner"] == 1)
    d = sum(1 for r in rows if r["winner"] == "draw")
    t = sum(1 for r in rows if r["winner"] == "timeout")
    lo, hi = wilson_ci(p0, n)
    return dict(
        label=label, n_games=n,
        p0_wins=p0, p1_wins=p1, draws=d, timeouts=t,
        p0_win_rate=round(p0/n if n else 0, 4),
        p0_win_rate_wilson_95_ci=[round(lo,4), round(hi,4)],
        termination_rate=round(sum(1 for r in rows if r["winner"]!="timeout")/n, 4) if n else 0,
        avg_actions_per_game=round(statistics.mean(r["actions"] for r in rows), 1) if rows else 0,
        avg_knockouts_per_game=round(statistics.mean(r["knockouts"] for r in rows), 1) if rows else 0,
        avg_decision_ms_p0=round(statistics.mean(r["avg_ms_p0"] for r in rows if r["avg_ms_p0"]>0), 1) if any(r["avg_ms_p0"]>0 for r in rows) else 0,
    )

OUT = pathlib.Path("/kaggle/working/eval")
OUT.mkdir(parents=True, exist_ok=True)

rows_ar = run_match("A: Trained Agent vs Random",
                     DECK, make_agent_pol(agent_trained), random_pol,
                     EVAL_N_GAMES_AGENT_VS_RANDOM, EVAL_MAX_ACTIONS_PER_GAME)
rows_ah = run_match("B: Trained Agent vs Heuristic",
                     DECK, make_agent_pol(agent_trained), make_agent_pol(agent_heur),
                     EVAL_N_GAMES_AGENT_VS_HEURISTIC, EVAL_MAX_ACTIONS_PER_GAME)
rows_mirror = run_match("C: Trained vs Trained (mirror)",
                         DECK, make_agent_pol(agent_trained), make_agent_pol(agent_trained_b),
                         max(EVAL_N_GAMES_AGENT_VS_HEURISTIC // 2, 6),
                         EVAL_MAX_ACTIONS_PER_GAME)

summaries = {
    "deck": {"top_attackers": TOP5, "basic_energy": ENERGY,
             "pokemon_count": 20, "energy_count": 40},
    "benchmark": BENCH,
    "checkpoint": str(ASSERT_CKPT) if ASSERT_CKPT else None,
    "training": {
        "rounds_completed": result.rounds_completed,
        "promotions": result.promotions,
        "elapsed_s": result.elapsed_s,
    },
    "matches": [
        summarize("trained_vs_random", rows_ar),
        summarize("trained_vs_heuristic", rows_ah),
        summarize("trained_vs_trained_mirror", rows_mirror),
    ],
}

for name, rows in [("trained_vs_random", rows_ar),
                    ("trained_vs_heuristic", rows_ah),
                    ("trained_vs_trained_mirror", rows_mirror)]:
    if not rows: continue
    with (OUT / f"{name}.csv").open("w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=list(rows[0].keys()))
        w.writeheader(); w.writerows(rows)
(OUT / "summary.json").write_text(json.dumps(summaries, indent=2, default=str))
print("\nSUMMARY:")
for s in summaries["matches"]:
    print(f"  {s['label']:<28} p0_win={s['p0_win_rate']:.1%} "
          f"CI={s['p0_win_rate_wilson_95_ci']} "
          f"term={s['termination_rate']:.0%} "
          f"avg_act={s['avg_actions_per_game']}")

## 7. Figures

In [ ]:
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

FIG = pathlib.Path("/kaggle/working/figures"); FIG.mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi": 130, "savefig.dpi": 160})

# A: win rate per match with CIs
labels = [s["label"] for s in summaries["matches"]]
rates  = [s["p0_win_rate"] for s in summaries["matches"]]
lows   = [r - s["p0_win_rate_wilson_95_ci"][0] for r, s in zip(rates, summaries["matches"])]
highs  = [s["p0_win_rate_wilson_95_ci"][1] - r for r, s in zip(rates, summaries["matches"])]
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(labels, rates, xerr=[lows, highs], color=["#5E5CFF","#00C49F","#FFB623"], capsize=4)
ax.set_xlim(0, 1); ax.set_xlabel("P0 win rate (95% CI)")
ax.set_title("Trained agent — head-to-head matchups")
for i, r in enumerate(rates):
    ax.text(r + 0.02, i, f"{r:.0%}", va="center")
plt.tight_layout(); fig.savefig(FIG/"win_rates.png"); plt.close()

# B: termination + KOs per match
term = [s["termination_rate"] for s in summaries["matches"]]
kos  = [s["avg_knockouts_per_game"] for s in summaries["matches"]]
fig, ax = plt.subplots(figsize=(8, 4))
x = list(range(len(labels))); w = 0.4
ax.bar([i-w/2 for i in x], term, w, color="#5E5CFF", label="termination rate")
ax2 = ax.twinx()
ax2.bar([i+w/2 for i in x], kos, w, color="#FF647C", label="avg KOs")
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=15)
ax.set_ylim(0, 1.05); ax.set_ylabel("termination rate")
ax2.set_ylabel("average KOs per game")
ax.set_title("Game terminality and combat intensity")
plt.tight_layout(); fig.savefig(FIG/"termination_kos.png"); plt.close()

# C: training summary
fig, ax = plt.subplots(figsize=(8, 4))
ax.axis("off")
rows_t = [
    ["rounds completed", result.rounds_completed],
    ["promotions", result.promotions],
    ["elapsed (s)", round(result.elapsed_s, 1)],
    ["selfplay games / round", SELFPLAY_GAMES_PER_ROUND],
    ["selfplay MCTS iter", MCTS_ITERATIONS_SELFPLAY],
    ["checkpoint", str(ASSERT_CKPT.name) if ASSERT_CKPT else "(none)"],
]
tab = ax.table(cellText=rows_t, colLabels=["metric", "value"],
                cellLoc="left", loc="center")
tab.auto_set_font_size(False); tab.set_fontsize(11); tab.scale(1, 1.4)
ax.set_title("Training summary", pad=12)
plt.tight_layout(); fig.savefig(FIG/"training_summary.png"); plt.close()
print("figures saved to", FIG)

## 8. Package outputs for download

Everything you need to attach to the Kaggle Writeup will be under `/kaggle/working/` after this cell. Right-click → Download in the Output tab.

In [ ]:
SUBMIT = pathlib.Path("/kaggle/working/submission_t4x2"); SUBMIT.mkdir(exist_ok=True)
# Copy checkpoint
if ASSERT_CKPT and ASSERT_CKPT.exists():
    subprocess.run(["cp", str(ASSERT_CKPT), str(SUBMIT / ASSERT_CKPT.name)], check=True)
# Copy eval + figures
subprocess.run(["cp", "-r", str(OUT), str(SUBMIT / "eval")], check=False)
subprocess.run(["cp", "-r", str(FIG), str(SUBMIT / "figures")], check=False)
# Manifest
manifest = {
    "checkpoint": ASSERT_CKPT.name if ASSERT_CKPT else None,
    "benchmark": BENCH,
    "summary": summaries,
    "environment": {
        "torch": torch.__version__,
        "cuda": torch.cuda.is_available(),
        "device_count": torch.cuda.device_count(),
        "device_names": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
    },
    "reproduce": {
        "training_rounds": TRAINING_ROUNDS,
        "selfplay_games_per_round": SELFPLAY_GAMES_PER_ROUND,
        "mcts_iter_selfplay": MCTS_ITERATIONS_SELFPLAY,
        "mcts_iter_arena": MCTS_ITERATIONS_ARENA,
        "eval_iter": EVAL_AGENT_ITERATIONS,
    },
}
(SUBMIT/"MANIFEST.json").write_text(json.dumps(manifest, indent=2, default=str))
print("\nContents of", SUBMIT)
subprocess.run(["ls", "-laR", str(SUBMIT)])
print("\n→ Download /kaggle/working/submission_t4x2/ from the Output panel.")

## 9. What to attach to the Kaggle Writeup

After this notebook finishes, your Writeup gains the three pieces that were missing:

1. **A trained checkpoint** — `submission_t4x2/ckpt_*.pt`.
2. **Real win-rate numbers** — `submission_t4x2/eval/summary.json` plus three CSVs.
3. **Three new figures** — `submission_t4x2/figures/win_rates.png`, `termination_kos.png`, `training_summary.png`.

Paste the win-rate numbers into the writeup's *Evaluation* section, attach the three figures to the Media Gallery, and the Model and Report scores both move up materially.